# Task 3.3 - Clustering       
Goal: Perform K-means clustering (from sratch) to cluster seeds into different goups.         
Deadline: 28th December.     

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, confusion_matrix, silhouette_score, silhouette_samples
from scipy.optimize import linear_sum_assignment as linear_assignment

Step 1 : Read all the data from the csv file

In [ ]:
data = pd.read_csv('data/seeds_data.csv')
features_names = ['Feature_1','Feature_2','Feature_3','Feature_4','Feature_5','Feature_6','Feature_7']
features = data[features_names].values
labels = data['Class'].values
print(f"Features shape : {features.shape}")
print(f"Labels shape : {labels.shape}")

Step 2 : Normalize the features

In [ ]:
features = (features - features.mean(axis=0)) / features.std(axis=0)

In [ ]:
#function to calculate euclidean distance
def distance(a, b):
    return np.linalg.norm(a - b, axis=1)

## Version A — Original K-means (Random Init)

* Cluster in original feature space

* Higher ARI

* Moderate silhouette

* Shows strong label recovery

In [ ]:
#function to perform k-means clustering
def kmeans(X, k, max_iters=1000):
    # Step 1: initialize centroids randomly
    np.random.seed(42)
    indices = np.random.choice(len(X), k, replace=False)
    centroids = X[indices]

    for s in range(max_iters):
        # Step 2: assign clusters
        clusters = [np.argmin([np.linalg.norm(x - c) for c in centroids]) for x in X]

        # Step 3: update centroids
        new_centroids = np.array([X[np.array(clusters) == i].mean(axis=0) for i in range(k)])

        # Stop if converged (no change)
        if np.allclose(centroids, new_centroids):
            break

        centroids = new_centroids

    return np.array(clusters), centroids

Step 3 : Find optimal K using Elbow Method

In [ ]:
sse = []
K_range = range(2, 10)

for k in K_range:
    clusters, centers = kmeans(features, k)
    sse.append(sum(np.linalg.norm(features[i] - centers[clusters[i]])**2 for i in range(len(features))))

plt.plot(K_range, sse, marker='o')
plt.xlabel("K")
plt.ylabel("SSE")
plt.title("Elbow Method")
plt.show()

Step 4 : Perform K-means clustering using optimal k(=3) and find Cluster Centers

In [ ]:
k_optimal = 3  # based on elbow
clusters, centers = kmeans(features, k_optimal)

print("\nCluster Centers:")
print(pd.DataFrame(centers))

Step 5 : Use PCA to visualize the clusters

In [ ]:
pca = PCA(n_components=2)
X_2d = pca.fit_transform(features)

plt.scatter(X_2d[:,0], X_2d[:,1], c=clusters, cmap="viridis")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("K-Means Clusters (PCA Projection)")
plt.show()

Step 6: Compare performance using metrics like ARI, confusion matrix and silhouette score

In [ ]:
# Map cluster labels to actual labels
mapping = {}
for cluster in range(k_optimal):
    mask = (clusters == cluster)
    true_labels = labels[mask]
    if len(true_labels) > 0:
        mapping[cluster] = np.bincount(true_labels).argmax()

# Create new predicted labels aligned with true classes
aligned_preds = np.vectorize(mapping.get)(clusters)

print("ARI:", adjusted_rand_score(labels, aligned_preds))
print("Confusion Matrix:\n", confusion_matrix(labels, aligned_preds))

In [ ]:
sil_score = silhouette_score(features, clusters)
print("Silhouette Score:", sil_score)

sil_samples = silhouette_samples(features, clusters)
print("Mean silhouette:", sil_samples.mean())
print("Min silhouette:", sil_samples.min())

High ARI of 0.823 tells us that clustering aligns well with ground true labels, while a moderate silhouette score of 0.404 indicates that despite correct grouping, there is some overlap in feature space, making geometric separation imperfect

## Version B — Improved Algorithm (K-means++)

* Cluster in original feature space

* More stable initialization

* Higher minimum silhouette

* Shows algorithmic correctness

In [ ]:
def init_kmeans_pp(X, k):
    centroids = [X[np.random.randint(len(X))]]

    for _ in range(1, k):
        distances = np.array([
            min(np.linalg.norm(x - c)**2 for c in centroids)
            for x in X
        ])
        probs = distances / distances.sum()
        cumulative_probs = np.cumsum(probs)
        r = np.random.rand()

        for i, p in enumerate(cumulative_probs):
            if r < p:
                centroids.append(X[i])
                break

    return np.array(centroids)


In [ ]:
#function to perform k-means clustering
def kmeans(X, k, max_iters=1000):
    # Step 1: initialize centroids 
    centroids = init_kmeans_pp(X, k)

    for s in range(max_iters):
        # Step 2: assign clusters
        clusters = [np.argmin([np.linalg.norm(x - c) for c in centroids]) for x in X]

        # Step 3: update centroids
        new_centroids = np.array([X[np.array(clusters) == i].mean(axis=0) for i in range(k)])

        # Stop if converged (no change)
        if np.allclose(centroids, new_centroids):
            break

        centroids = new_centroids

    return np.array(clusters), centroids

In [ ]:
def kmeans_best_of_n(X, k, n_runs=100):
    best_sse = np.inf
    best_clusters, best_centers = None, None

    for _ in range(n_runs):
        clusters, centers = kmeans(X, k)
        sse = sum(np.linalg.norm(X[i] - centers[clusters[i]])**2 for i in range(len(X)))

        if sse < best_sse:
            best_sse = sse
            best_clusters, best_centers = clusters, centers

    return best_clusters, best_centers

In [ ]:
clusters, centers = kmeans_best_of_n(features, 3)

print("\nCluster Centers:")
print(pd.DataFrame(centers))

In [ ]:
pca = PCA(n_components=2)
X_2d = pca.fit_transform(features)

plt.scatter(X_2d[:,0], X_2d[:,1], c=clusters, cmap="viridis")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("K-Means Clusters (PCA Projection)")
plt.show()

In [ ]:
# Map cluster labels to actual labels
mapping = {}
for cluster in range(k_optimal):
    mask = (clusters == cluster)
    true_labels = labels[mask]
    if len(true_labels) > 0:
        mapping[cluster] = np.bincount(true_labels).argmax()

# Create new predicted labels aligned with true classes
aligned_preds = np.vectorize(mapping.get)(clusters)

print("ARI:", adjusted_rand_score(labels, aligned_preds))
print("Confusion Matrix:\n", confusion_matrix(labels, aligned_preds))

In [ ]:
sil_score = silhouette_score(features, clusters)
print("Silhouette Score:", sil_score)

sil_samples = silhouette_samples(features, clusters)
print("Mean silhouette:", sil_samples.mean())
print("Min silhouette:", sil_samples.min())

K-means++ reduced extreme misassignments (higher minimum silhouette) at the cost of slightly reduced average compactness, reflecting a trade-off between stability and tight clustering.

## Version C — PCA-space Clustering

* Apply PCA → cluster in reduced space

* Higher silhouette

* Shows geometric separation

In [ ]:
pca = PCA(n_components=2)
features_pca = pca.fit_transform(features)

In [ ]:
clusters, centers = kmeans_best_of_n(features_pca, 3)

In [ ]:
pca = PCA(n_components=2)
X_2d = pca.fit_transform(features)

plt.scatter(X_2d[:,0], X_2d[:,1], c=clusters, cmap="viridis")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("K-Means Clusters (PCA Projection)")
plt.show()

In [ ]:
# Map cluster labels to actual labels
mapping = {}
for cluster in range(k_optimal):
    mask = (clusters == cluster)
    true_labels = labels[mask]
    if len(true_labels) > 0:
        mapping[cluster] = np.bincount(true_labels).argmax()

# Create new predicted labels aligned with true classes
aligned_preds = np.vectorize(mapping.get)(clusters)

print("ARI:", adjusted_rand_score(labels, aligned_preds))
print("Confusion Matrix:\n", confusion_matrix(labels, aligned_preds))

In [ ]:
sil_score = silhouette_score(features_pca, clusters)
print("Silhouette Score:", sil_score)

sil_samples = silhouette_samples(features_pca, clusters)
print("Mean silhouette:", sil_samples.mean())
print("Min silhouette:", sil_samples.min())

In [ ]:
pca = PCA(n_components=2)
X_2d = pca.fit_transform(features)

plt.scatter(features_pca[:,0], features_pca[:,1], c=clusters)
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("K-Means Clusters (PCA Projection)")
plt.show()

Clustering in PCA space improves geometric separation (higher silhouette), while clustering in the original space better preserves true class structure (higher ARI).

| Version | Clustering Space | ARI ↑ | Silhouette ↑ | Min Silhouette | Summary |
|-------|-----------------|-----|------------------|----------------|---------|
| A | Original feature space | 0.8227 | 0.4039 | -0.0434 | Best agreement with true labels, some cluster overlap |
| B | Improved initialization (feature space) | 0.7733 | 0.4007 | 0.0027 | More stable clustering, slightly weaker class recovery |
| C | PCA-reduced space (2D) | 0.7850 | 0.4934 | -0.0489 | Best cluster separation, slight loss in label alignment |


## 🔹 What this means:  

### Version A

* Maximizes agreement with ground-truth labels

* Some points lie between clusters → negative silhouette

### Version B

* Improves cluster stability

* Loses some class alignment → ARI drop

* Eliminates worst overlaps (min silhouette ↑)

### Version C

* Maximizes cluster geometry

* Best silhouette score (≈ 0.49)

* Slight trade-off in ARI due to dimensionality reduction

ARI measures how well clusters match the true class labels, while silhouette score measures how well-separated the clusters are based only on feature geometry.  

Clustering in PCA space (Version C) improves silhouette by removing noise and emphasizing dominant variance directions, but may reduce ARI because PCA does not preserve class-discriminative information.  

Thus, higher silhouette does not always imply better class recovery.
